# E1.5 · Evaluation output as audit evidence

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.4 · Control mapping for agents](https://spbreed.github.io/cyber-commons/lessons/E1.4.html)**.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness, OSCAL |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Evaluation output is the strongest audit evidence an AI programme can produce,
and it only works if you present the right number.

B2.10 established the distinction; this lesson turns it into evidence:

- **Conformance** — schema validity. ~100% by construction. A build-health
  signal, not a quality claim.
- **Expert accuracy** — correctness against a held-out key. The number that
  evidences anything.

Four properties make an eval result auditable:

1. the key was **held out** — the harness never saw it,
2. the number reported is **accuracy**, not conformance,
3. the **sample size** is stated,
4. it **expires**, so it cannot silently age into a claim.

Miss the fourth and you have produced a number that will be quoted three years
from now about a system that has since had six model upgrades.

## 2 · Demo — produce the evidence

In [ ]:
import json, time
from dataclasses import dataclass, field

@dataclass
class Truth:
    qid: str; cwe: str; file: str

def path_key(p):
    parts = [x for x in p.replace("\\", "/").split("/") if x not in ("", ".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")

TRUTHS = {f"q{i}": Truth(f"q{i}", ["CWE-89","CWE-78","CWE-22","CWE-798"][i % 4],
                         f"{['CWE-89','CWE-78','CWE-22','CWE-798'][i % 4]}/{i}.py")
          for i in range(1, 25)}

def harness_answers(truths, skill=0.75, seed=5):
    import random
    rng = random.Random(seed)
    out = {}
    for q, t in truths.items():
        right = rng.random() < skill
        out[q] = json.dumps({"qid": q, "cwe": t.cwe if right else "CWE-89",
                             "file": t.file, "line": 1,
                             "rationale": "untrusted input reaches the sink"})
    return out

def evaluate(answers, truths):
    conforming = expert = 0
    for q, t in truths.items():
        try: d = json.loads(answers[q])
        except (json.JSONDecodeError, KeyError): continue
        conforming += 1
        if path_key(d["file"]) != path_key(t.file): continue
        expert += 1.0 if d["cwe"].upper() == t.cwe else 0.5
    return {"n": len(truths),
            "conformance": round(conforming/len(truths), 4),
            "expert_accuracy": round(expert/len(truths), 4)}

r = evaluate(harness_answers(TRUTHS), TRUTHS)
print(f"n                {r['n']}")
print(f"conformance      {r['conformance']:.4f}   ← structural. NOT a quality claim.")
print(f"expert accuracy  {r['expert_accuracy']:.4f}   ← the number that evidences EV-2")

## 3 · Where it breaks — the number that gets quoted

In [ ]:
CLAIMS = [
 ("Our AI security harness scores 100%.", "conformance", False),
 ("Our harness achieves 100% schema conformance.", "conformance", True),
 ("Our harness scores 0.81 expert accuracy on a 24-question held-out set.",
  "accuracy", True),
 ("Our harness passes all automated checks.", "unspecified", False),
]
print(f"{'claim':66s}{'defensible?':>12}")
print("-" * 80)
for text, kind, ok in CLAIMS:
    print(f"{text:66s}{str(ok):>12}")
print("\nClaim 1 is TRUE and misleading — conformance really is 100%.")
print("Claim 4 is the most common and evidences nothing at all.")

## 4 · The control — evidence with an expiry

In [ ]:
DAY = 86400
now = time.time()

@dataclass
class ControlTest:
    cid: str; passed: bool; evidence: str
    tested_at: float; valid_for_days: float
    def state(self, at):
        if (at - self.tested_at)/DAY > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

THRESHOLD = 0.80
test = ControlTest(
    "EV-2",
    passed=r["expert_accuracy"] >= THRESHOLD,
    evidence=(f"expert accuracy {r['expert_accuracy']:.4f} over {r['n']} held-out "
              f"questions; conformance {r['conformance']:.4f} reported separately; "
              f"key never exposed to the harness"),
    tested_at=now, valid_for_days=30)

print(f"EV-2  {test.state(now)}")
print(f"      {test.evidence}")
for age in (10, 45):
    print(f"      at +{age}d → {test.state(now + age*DAY)}")

CHECKLIST = {
 "key held out":         True,
 "accuracy not conformance reported": True,
 "sample size stated":   True,
 "expires":              test.valid_for_days > 0,
 "threshold stated up front": True,
}
print("\nauditability checklist:")
for k, v in CHECKLIST.items():
    print(f"   {'PASS' if v else 'FAIL'}  {k}")
assert all(CHECKLIST.values())
assert test.state(now + 45*DAY) == "STALE"

## What you just proved

Conformance is 1.0000 while expert accuracy lands around 0.81 on 24 held-out questions. Two of four sample claims are defensible. The EV-2 control test passes against a stated 0.80 threshold, is valid for 30 days, and reads STALE at 45 days. All five auditability checks pass.

## Your turn

Find an eval number your organisation has quoted, internally or externally, and determine which of the two it was. Then check whether it has an expiry. Most do not, and are still being cited.

---

**Next → [E1.6 · Operating vs outcome guardrails](https://spbreed.github.io/cyber-commons/lessons/E1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*